# Heart disease XAI: corrected evaluation (patient-level CV, seeds, threshold tuning, Grad-CAM)

**Author:** Karthik H D (SCA25MSD038)

This notebook re-runs the EfficientNet-B0 model from the original notebook with the **same architecture,
schedule, callbacks and batch size**, but with a clean evaluation:

1. **Patient-level split.** Each `Directory_N` folder in the CAD Cardiac MRI Dataset is one patient
   (16 Normal, 14 Sick). Patients are dealt into 5 stratified folds. In each run one fold is the test set,
   2 Normal + 2 Sick patients from the other folds are the validation set, and the rest are training data.
   No patient is ever in two subsets of the same run.
2. **Five runs, one seed each** (42, 7, 123, 2024, 31). Results are reported as mean ± SD.
3. **Threshold tuning** on the validation patients for a target sensitivity (default 95%).
4. **Saved probabilities**, so ROC-AUC and any threshold can be recomputed later.
5. **Class counts** for the original image-level split (fills the "n/r" cells in Table I).
6. **Grad-CAM** maps for 8 random test images (2 TP, 2 TN, 2 FN, 2 FP), each explained by the model
   that never saw that patient.
7. Writes `paper_values.json`, `fig3_confusion.png` and `fig5_gradcam.png` for `fill_paper.py`.

**Runtime:** each run trains on ~42k images, about the same as the original (~3 h on a Colab GPU).
Five runs take ~15 h, so results are saved to Google Drive after every run. If Colab disconnects,
just run all cells again: finished runs are skipped. (A run interrupted mid-training restarts from scratch.)

Use **Runtime → Change runtime type → GPU** before starting.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, gc, json, time, zipfile, math
import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, models, optimizers, callbacks
from tensorflow.keras.applications import EfficientNetB0
from sklearn.metrics import roc_auc_score, roc_curve

print("TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices('GPU'))

# ---------------- configuration ----------------
ZIP_PATH    = "/content/drive/MyDrive/Heart_disease_image_dataset"  # same zip as the original notebook
DATA_DIR    = "/content/heart_data"
RESULTS_DIR = "/content/drive/MyDrive/heart_xai_results"            # on Drive so runs survive disconnects

IMG_SIZE      = (224, 224)
BATCH_SIZE    = 8
EPOCHS_STAGE1 = 25
EPOCHS_STAGE2 = 15
CLASS_NAMES   = ["Normal", "Sick"]   # 0 = Normal, 1 = Sick (positive class), same as the original

N_FOLDS     = 5
RUN_SEEDS   = [42, 7, 123, 2024, 31]   # one seed per run / fold
SPLIT_SEED  = 42                       # seed for assigning patients to folds
N_VAL_PATIENTS_PER_CLASS = 2
TARGET_SENS = 0.95                     # target sensitivity for threshold tuning

# The original run, for the side-by-side numbers (copied from the original notebook output)
ORIG = dict(tn=5566, fp=100, fn=135, tp=3711, auc=0.9978)

os.makedirs(RESULTS_DIR, exist_ok=True)
AUTOTUNE = tf.data.AUTOTUNE

## 1. Get the data
Uses the same zip on Drive as the original notebook. If it is not there, the dataset is downloaded
from Kaggle with `kagglehub` (needs a Kaggle account / API token).

In [ ]:
if not (os.path.isdir(DATA_DIR) and os.listdir(DATA_DIR)):
    os.makedirs(DATA_DIR, exist_ok=True)
    if os.path.exists(ZIP_PATH):
        print("Extracting zip from Drive (a few minutes for ~3 GB)...")
        with zipfile.ZipFile(ZIP_PATH, "r") as zf:
            zf.extractall(DATA_DIR)
    else:
        import kagglehub
        DATA_DIR = kagglehub.dataset_download("danialsharifrazi/cad-cardiac-mri-dataset")
    print("Done.")

def find_data_root(d):
    """Folder that directly contains the Normal/ and Sick/ class folders."""
    for root, dirs, _ in os.walk(d):
        if "Normal" in dirs and "Sick" in dirs:
            return root
    raise FileNotFoundError(f"No folder with Normal/ and Sick/ found under {d}")

DATA_ROOT = find_data_root(DATA_DIR)
print("Data root:", DATA_ROOT)

## 2. Index every image with its class and patient

In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".bmp", ".gif")   # same formats the Keras directory loader accepts
rows, loose = [], []
for label, cls in enumerate(CLASS_NAMES):
    cdir = os.path.join(DATA_ROOT, cls)
    for entry in sorted(os.listdir(cdir)):
        p = os.path.join(cdir, entry)
        if not os.path.isdir(p):
            if entry.lower().endswith(EXTS):
                loose.append(p)
            continue
        for root, _, files in os.walk(p):
            for f in sorted(files):
                if f.lower().endswith(EXTS):
                    rows.append((os.path.join(root, f), label, cls, f"{cls}/{entry}",
                                 os.path.relpath(root, p)))
df = pd.DataFrame(rows, columns=["path", "label", "cls", "patient", "series"])
if loose:
    print(f"WARNING: {len(loose)} images sit directly in a class folder (no patient folder); they are excluded.")

pat = (df.groupby("patient")
         .agg(label=("label", "first"), cls=("cls", "first"), images=("path", "size"),
              series=("series", "nunique"))
         .reset_index())
print(f"Images: {len(df):,} | Normal: {(df.label==0).sum():,} | Sick: {(df.label==1).sum():,}")
print(f"Patients: {len(pat)} | Normal: {(pat.label==0).sum()} | Sick: {(pat.label==1).sum()}")
print(f"Nested series folders: {df.groupby('patient').series.nunique().sum():,}")
print(pat.to_string(index=False))
print("\nExample file names (check that nothing in the names leaks the label):")
print(df.sample(5, random_state=0).path.str.replace(DATA_ROOT, "", regex=False).to_string(index=False))
df.to_csv(os.path.join(RESULTS_DIR, "image_index.csv"), index=False)

## 3. Class counts of the ORIGINAL image-level split (fills the "n/r" cells of Table I)
Re-runs exactly the original `image_dataset_from_directory` calls (30% split, seed 42) and reads
their file lists. The validation counts are "held-out pool minus the final test pass", because in the
original code the validation half changed every epoch (paper Section V-D).

In [ ]:
_kw = dict(validation_split=0.3, seed=42, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode="binary")
orig_tr = tf.keras.utils.image_dataset_from_directory(DATA_ROOT, subset="training", **_kw)
orig_ho = tf.keras.utils.image_dataset_from_directory(DATA_ROOT, subset="validation", **_kw)

def count_cls(paths):
    sick = sum(1 for p in paths if f"{os.sep}Sick{os.sep}" in p)
    return len(paths) - sick, sick

o_tr_n, o_tr_s = count_cls(orig_tr.file_paths)
o_ho_n, o_ho_s = count_cls(orig_ho.file_paths)
o_te_n, o_te_s = ORIG["tn"] + ORIG["fp"], ORIG["tp"] + ORIG["fn"]
o_va_n, o_va_s = o_ho_n - o_te_n, o_ho_s - o_te_s
print(f"Original training : {len(orig_tr.file_paths):,}  Normal {o_tr_n:,}  Sick {o_tr_s:,}")
print(f"Original held-out : {len(orig_ho.file_paths):,}  Normal {o_ho_n:,}  Sick {o_ho_s:,}")
print(f"Original validation (pool - test): Normal {o_va_n:,}  Sick {o_va_s:,}")
if len(orig_tr.file_paths) != 44398:
    print("NOTE: training size differs from the original 44,398, so the TF/Keras version shuffles "
          "differently. Report these counts with that caveat.")
del orig_tr, orig_ho

## 4. Patient-level folds

In [ ]:
rng = np.random.RandomState(SPLIT_SEED)
fold_of = {}
for lab in (0, 1):
    ids = pat.loc[pat.label == lab, "patient"].values.copy()
    rng.shuffle(ids)
    for i, pid in enumerate(ids):
        # Normal patients are dealt to folds 0,1,2,3,4,0,...; Sick in reverse order 4,3,2,1,0,4,...
        # so that every test fold gets 6 patients (16 Normal + 14 Sick).
        fold_of[pid] = (i % N_FOLDS) if lab == 0 else (N_FOLDS - 1 - i % N_FOLDS)
pat["fold"] = pat.patient.map(fold_of)

splits = {}
for k in range(N_FOLDS):
    r = np.random.RandomState(SPLIT_SEED + 100 + k)
    test_p = set(pat.loc[pat.fold == k, "patient"])
    val_p = set()
    for lab in (0, 1):
        cand = pat.loc[(pat.fold != k) & (pat.label == lab), "patient"].values
        val_p |= set(r.choice(cand, N_VAL_PATIENTS_PER_CLASS, replace=False))
    train_p = set(pat.patient) - test_p - val_p
    assert not (train_p & val_p) and not (train_p & test_p) and not (val_p & test_p)
    splits[k] = dict(train=sorted(train_p), val=sorted(val_p), test=sorted(test_p))

# every patient is tested exactly once
assert sorted(sum((splits[k]["test"] for k in splits), [])) == sorted(pat.patient)
with open(os.path.join(RESULTS_DIR, "patient_splits.json"), "w") as f:
    json.dump(splits, f, indent=1)

split_rows = []
for k, s in splits.items():
    for name in ("train", "val", "test"):
        sub = df[df.patient.isin(s[name])]
        split_rows.append(dict(fold=k, subset=name, patients=len(s[name]), images=len(sub),
                               normal=int((sub.label == 0).sum()), sick=int((sub.label == 1).sum())))
split_table = pd.DataFrame(split_rows)
split_table.to_csv(os.path.join(RESULTS_DIR, "split_counts.csv"), index=False)
print(split_table.to_string(index=False))

## 5. Input pipeline and model (same as the original)

In [ ]:
def load_image(path, label):
    img = tf.io.read_file(path)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE)           # bilinear, float 0-255, like the Keras directory loader
    img.set_shape(IMG_SIZE + (3,))
    return img, tf.reshape(label, (1,))

def make_ds(sub, shuffle=False, seed=None):
    ds = tf.data.Dataset.from_tensor_slices((sub.path.values, sub.label.values.astype("float32")))
    if shuffle:
        ds = ds.shuffle(len(sub), seed=seed, reshuffle_each_iteration=True)
    return ds.map(load_image, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

preprocess_input = tf.keras.applications.efficientnet.preprocess_input

def build_model():
    augment = tf.keras.Sequential([
        layers.RandomFlip("horizontal"),
        layers.RandomRotation(0.05),
        layers.RandomZoom(0.1),
        layers.RandomContrast(0.1),
    ], name="data_augmentation")
    base = EfficientNetB0(include_top=False, weights="imagenet", input_shape=IMG_SIZE + (3,))
    base.trainable = False
    inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
    x = augment(inputs)
    x = preprocess_input(x)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D(name="gap")(x)
    x = layers.Dropout(0.3, name="head_dropout_1")(x)
    x = layers.Dense(128, activation="relu", name="head_dense")(x)
    x = layers.Dropout(0.2, name="head_dropout_2")(x)
    outputs = layers.Dense(1, activation="sigmoid", name="prediction")(x)
    return models.Model(inputs, outputs), base

def compile_model(model, lr):
    model.compile(optimizer=optimizers.Adam(learning_rate=lr), loss="binary_crossentropy",
                  metrics=["accuracy", tf.keras.metrics.AUC(name="auc"),
                           tf.keras.metrics.Precision(name="precision"),
                           tf.keras.metrics.Recall(name="recall")])

## 6. Train the five runs (skips runs that are already finished)

In [ ]:
def run_dir(k):
    return os.path.join(RESULTS_DIR, f"run{k}_seed{RUN_SEEDS[k]}")

def train_run(k):
    out = run_dir(k)
    if os.path.exists(os.path.join(out, "test_probs.npy")):
        print(f"Run {k} already finished, skipping."); return
    os.makedirs(out, exist_ok=True)
    tf.keras.backend.clear_session(); gc.collect()
    seed = RUN_SEEDS[k]
    tf.keras.utils.set_random_seed(seed)          # Python, NumPy and TensorFlow seeds

    s = splits[k]
    tr, va, te = (df[df.patient.isin(s[n])].reset_index(drop=True) for n in ("train", "val", "test"))
    train_ds = make_ds(tr, shuffle=True, seed=seed)
    val_ds, test_ds = make_ds(va), make_ds(te)

    model, base = build_model()
    compile_model(model, 1e-3)
    cb = [callbacks.EarlyStopping(monitor="val_auc", mode="max", patience=6, restore_best_weights=True),
          callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.3, patience=3),
          callbacks.ModelCheckpoint(os.path.join(out, "best_model.keras"),
                                    monitor="val_auc", mode="max", save_best_only=True)]
    t0 = time.time()
    h1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE1, callbacks=cb)

    base.trainable = True
    for layer in base.layers[:len(base.layers) - 40]:
        layer.trainable = False
    compile_model(model, 1e-5)
    h2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_STAGE2, callbacks=cb)
    hours = (time.time() - t0) / 3600

    model.save(os.path.join(out, "final_model.keras"))
    np.save(os.path.join(out, "val_probs.npy"), model.predict(val_ds, verbose=0).ravel())
    np.save(os.path.join(out, "test_probs.npy"), model.predict(test_ds, verbose=0).ravel())
    hist = {"stage1": {m: [float(v) for v in vals] for m, vals in h1.history.items()},
            "stage2": {m: [float(v) for v in vals] for m, vals in h2.history.items()},
            "hours": hours, "seed": seed}
    with open(os.path.join(out, "history.json"), "w") as f:
        json.dump(hist, f)
    print(f"Run {k} done in {hours:.2f} h")

for k in range(N_FOLDS):
    train_run(k)

## 7. Metrics: per run, then mean ± SD

In [ ]:
def confusion(y, p, t):
    yp = (p >= t).astype(int)
    tp = int(((yp == 1) & (y == 1)).sum()); tn = int(((yp == 0) & (y == 0)).sum())
    fp = int(((yp == 1) & (y == 0)).sum()); fn = int(((yp == 0) & (y == 1)).sum())
    return tn, fp, fn, tp

def measures(tn, fp, fn, tp, auc=np.nan):
    tn, fp, fn, tp = map(float, (tn, fp, fn, tp)); n = tn + fp + fn + tp
    sens = tp / (tp + fn) if tp + fn else np.nan
    spec = tn / (tn + fp) if tn + fp else np.nan
    ppv  = tp / (tp + fp) if tp + fp else np.nan
    npv  = tn / (tn + fn) if tn + fn else np.nan
    den  = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    pe   = ((tp + fp) * (tp + fn) + (tn + fn) * (tn + fp)) / n ** 2
    lrp  = sens / (1 - spec) if spec < 1 else np.inf
    lrn  = (1 - sens) / spec if spec > 0 else np.inf
    return dict(acc=100 * (tp + tn) / n, sens=100 * sens, spec=100 * spec, ppv=100 * ppv, npv=100 * npv,
                f1=100 * 2 * tp / (2 * tp + fp + fn), bacc=100 * (sens + spec) / 2,
                mcc=(tp * tn - fp * fn) / den if den else np.nan,
                kappa=((tp + tn) / n - pe) / (1 - pe), j=sens + spec - 1,
                lrp=lrp, lrn=lrn, dor=lrp / lrn if lrn > 0 else np.inf, auc=auc)

def tune_threshold(y_val, p_val, target):
    """Highest threshold that keeps validation sensitivity >= target (predict Sick if p >= t)."""
    ps = np.sort(p_val[y_val == 1])
    return float(ps[int(np.floor((1 - target) * len(ps)))])

per_run, oof = [], []
for k in range(N_FOLDS):
    out = run_dir(k); s = splits[k]
    va = df[df.patient.isin(s["val"])].reset_index(drop=True)
    te = df[df.patient.isin(s["test"])].reset_index(drop=True)
    pv, pt = np.load(os.path.join(out, "val_probs.npy")), np.load(os.path.join(out, "test_probs.npy"))
    yv, yt = va.label.values, te.label.values
    thr = tune_threshold(yv, pv, TARGET_SENS)
    auc = roc_auc_score(yt, pt)
    m05 = measures(*confusion(yt, pt, 0.5), auc=auc)
    mt  = measures(*confusion(yt, pt, thr), auc=auc)
    hours = json.load(open(os.path.join(out, "history.json")))["hours"]
    per_run.append(dict(run=k, seed=RUN_SEEDS[k], thr=thr, hours=hours,
                        val_sens_at_thr=100 * (pv[yv == 1] >= thr).mean(),
                        **{f"{m}@0.5": v for m, v in m05.items()},
                        **{f"{m}@thr": v for m, v in mt.items()}))
    te = te.assign(prob=pt, run=k, thr=thr)
    oof.append(te)

per_run = pd.DataFrame(per_run)
oof = pd.concat(oof, ignore_index=True)
per_run.to_csv(os.path.join(RESULTS_DIR, "per_run_metrics.csv"), index=False)
oof.to_csv(os.path.join(RESULTS_DIR, "out_of_fold_predictions.csv"), index=False)
pd.set_option("display.width", 200)
print(per_run.round(4).T)

## 8. Figures: confusion matrices (Fig. 3) and ROC curves

In [ ]:
def draw_cm(ax, cm, title):
    cm = np.asarray(cm); row = cm / cm.sum(axis=1, keepdims=True)
    ax.imshow(row, cmap="Blues", vmin=0, vmax=1)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{cm[i, j]:,}\n({100 * row[i, j]:.2f}%)", ha="center", va="center",
                    fontsize=7, color="white" if row[i, j] > 0.5 else "black", family="serif")
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(CLASS_NAMES, fontsize=7, family="serif")
    ax.set_yticklabels(CLASS_NAMES, fontsize=7, family="serif", rotation=90, va="center")
    ax.set_xlabel("Predicted", fontsize=7, family="serif"); ax.set_ylabel("True", fontsize=7, family="serif")
    ax.set_title(title, fontsize=7.5, family="serif")
    for sp in ax.spines.values(): sp.set_linewidth(0.5)
    ax.tick_params(length=0)

y_all, p_all = oof.label.values, oof.prob.values
cm_new = np.array(confusion(y_all, p_all, 0.5)).reshape(2, 2)
cm_orig = np.array([[ORIG["tn"], ORIG["fp"]], [ORIG["fn"], ORIG["tp"]]])

fig, axes = plt.subplots(1, 2, figsize=(3.5, 1.95), dpi=300)
draw_cm(axes[0], cm_orig, "(a) Image-level split")
draw_cm(axes[1], cm_new, "(b) Patient-level CV")
plt.tight_layout(pad=0.3, w_pad=1.0)
fig.savefig(os.path.join(RESULTS_DIR, "fig3_confusion.png"), dpi=300); plt.show()

plt.figure(figsize=(3.5, 3), dpi=200)
for k in range(N_FOLDS):
    d = oof[oof.run == k]; fpr, tpr, _ = roc_curve(d.label, d.prob)
    plt.plot(fpr, tpr, lw=1, label=f"Run {k + 1} (AUC {per_run.loc[k, 'auc@0.5']:.3f})")
plt.plot([0, 1], [0, 1], "--", color="gray", lw=0.8)
plt.xlabel("False positive rate"); plt.ylabel("True positive rate"); plt.legend(fontsize=6)
plt.title("Patient-level CV, test folds", fontsize=8); plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, "roc_patient_level.png")); plt.show()

## 9. Grad-CAM (Fig. 5)
Eight test images drawn at random: 2 true positives, 2 true negatives, 2 false negatives, 2 false
positives (threshold 0.5), each from a different patient where possible. Each image is explained by the
model of the run in which its patient was in the test fold.

In [ ]:
def gradcam(model, img):
    """Grad-CAM on the 7x7x1280 top_activation output. Explains the predicted class.
    The head (GAP -> dense 128 ReLU -> dense 1) is applied with its own weights inside the tape; calling the
    Keras layers there loses the gradient in Keras 3. Dropout is the identity at inference, so this is exact."""
    base = next(l for l in model.layers if l.name.startswith("efficientnet"))
    dense, pred = model.get_layer("head_dense"), model.get_layer("prediction")
    x = preprocess_input(tf.convert_to_tensor(img[None], tf.float32))
    conv = tf.convert_to_tensor(base(x, training=False))
    with tf.GradientTape() as tape:
        tape.watch(conv)
        h = tf.nn.relu(tf.matmul(tf.reduce_mean(conv, axis=(1, 2)), dense.kernel) + dense.bias)
        logit = tf.matmul(h, pred.kernel) + pred.bias
        prob = float(tf.sigmoid(logit)[0, 0])
        score = logit[:, 0] if prob >= 0.5 else -logit[:, 0]   # pre-sigmoid score of the predicted class
    grads = tape.gradient(score, conv)
    weights = tf.reduce_mean(grads, axis=(1, 2))                 # alpha_k, Eq. (2)
    cam = tf.nn.relu(tf.reduce_sum(conv * weights[:, None, None, :], axis=-1))[0]   # Eq. (3)
    cam = (cam / (tf.reduce_max(cam) + 1e-8)).numpy()
    return cam, prob

def read_img(path):
    return load_image(tf.constant(path), tf.constant(0.0))[0].numpy()

def overlay(img, cam, alpha=0.4):
    cam_up = tf.image.resize(cam[..., None], IMG_SIZE, method="bilinear").numpy()[..., 0]
    heat = plt.get_cmap("jet")(cam_up)[..., :3] * 255
    return np.clip(img * (1 - alpha) + heat * alpha, 0, 255).astype("uint8")

oof["pred"] = (oof.prob >= 0.5).astype(int)
groups = [("True positive", 1, 1), ("True negative", 0, 0), ("False negative", 1, 0), ("False positive", 0, 1)]
rs = np.random.RandomState(SPLIT_SEED)
chosen = []
for name, y, yhat in groups:
    cand = oof[(oof.label == y) & (oof.pred == yhat)]
    picks, used = [], set()
    for idx in rs.permutation(len(cand)):
        row = cand.iloc[idx]
        if row.patient in used and len(cand.patient.unique()) > 1:
            continue
        picks.append(row); used.add(row.patient)
        if len(picks) == 2: break
    if len(picks) < 2:
        print(f"Only {len(picks)} example(s) available for {name}.")
    chosen += [(name, r) for r in picks]

model_cache = {}
fig, axes = plt.subplots(4, 2, figsize=(3.5, 6.4), dpi=300)
for ax in axes.flat: ax.axis("off")
for i, (name, r) in enumerate(chosen):
    k = int(r.run)
    if k not in model_cache:
        model_cache.clear(); tf.keras.backend.clear_session(); gc.collect()
        model_cache[k] = tf.keras.models.load_model(os.path.join(run_dir(k), "final_model.keras"))
    img = read_img(r.path)
    cam, prob = gradcam(model_cache[k], img)
    assert abs(prob - r.prob) < 1e-3, "Grad-CAM forward pass does not match the saved prediction"
    ax = axes[["True positive", "True negative", "False negative", "False positive"].index(name),
              sum(1 for n, _ in chosen[:i] if n == name)]
    ax.imshow(overlay(img, cam))
    ax.set_title(f"{name}\nTrue {CLASS_NAMES[r.label]} · Pred {CLASS_NAMES[int(prob >= 0.5)]} · p={prob:.2f}",
                 fontsize=6, family="serif")
plt.tight_layout(pad=0.3, h_pad=0.6, w_pad=0.4)
fig.savefig(os.path.join(RESULTS_DIR, "fig5_gradcam.png"), dpi=300); plt.show()
pd.DataFrame([dict(group=n, path=r.path, patient=r.patient, run=int(r.run) + 1, prob=r.prob)
              for n, r in chosen]).to_csv(os.path.join(RESULTS_DIR, "gradcam_examples.csv"), index=False)

## 10. Write `paper_values.json` for `fill_paper.py`

In [ ]:
def ms(col, fmt):
    v = per_run[col].replace([np.inf, -np.inf], np.nan).dropna()
    if len(v) == 0:
        return "—"
    txt = f"{v.mean():{fmt}} ± {v.std(ddof=1):{fmt}}" if len(v) > 1 else f"{v.mean():{fmt}}"
    return txt if len(v) == len(per_run) else txt + f" ({len(v)} of {len(per_run)} runs)"   # undefined in some runs, e.g. LR+ at 100% specificity

def c(n):
    return f"{int(round(n)):,}"

V = {}
fmts = dict(acc=".2f", sens=".2f", spec=".2f", ppv=".2f", npv=".2f", f1=".2f", bacc=".2f",
            mcc=".3f", kappa=".3f", j=".3f", lrp=".1f", lrn=".3f", dor=".0f", auc=".4f")
for m, f in fmts.items():
    V[f"NEW05_{m.upper()}"] = ms(f"{m}@0.5", f)
    V[f"NEWT_{m.upper()}"] = ms(f"{m}@thr", f)
V["THR"] = ms("thr", ".3f")
V["TARGET_SENS"] = f"{100 * TARGET_SENS:.0f}"
V["N_SEEDS"] = str(N_FOLDS)
V["NT_MET"] = str(int((per_run["sens@thr"] >= 100 * TARGET_SENS).sum()))
V["RUN_HOURS"] = f"{per_run.hours.mean():.1f}"

# Table I
V.update(O_TR_N=c(o_tr_n), O_TR_S=c(o_tr_s), O_VA_N=c(o_va_n), O_VA_S=c(o_va_s))
for name, key in (("train", "TR"), ("val", "VA"), ("test", "TE")):
    t = split_table[split_table.subset == name]
    V[f"C_{key}"] = c(t.images.mean()); V[f"C_{key}_N"] = c(t.normal.mean()); V[f"C_{key}_S"] = c(t.sick.mean())

# Fig. 3(b), pooled out-of-fold at 0.5
(tn, fp), (fn, tp) = cm_new
V.update(CM_TN=c(tn), CM_FP=c(fp), CM_FN=c(fn), CM_TP=c(tp), CM_ACC=f"{100 * (tn + tp) / cm_new.sum():.2f}",
         CM_MISS=f"{100 * fn / (fn + tp):.2f}", CM_FA=f"{100 * fp / (fp + tn):.2f}")

# Table IV, from mean sensitivity / specificity at the tuned threshold
se, sp = per_run["sens@thr"].mean() / 100, per_run["spec@thr"].mean() / 100
prev_data = (df.label == 1).mean()
for key, prev in (("1", .01), ("5", .05), ("10", .10), ("20", .20), ("D", prev_data)):
    ppv = se * prev / (se * prev + (1 - sp) * (1 - prev))
    npv = sp * (1 - prev) / (sp * (1 - prev) + (1 - se) * prev)
    V[f"T4_{key}_PPV"] = f"{100 * ppv:.1f}"; V[f"T4_{key}_NPV"] = f"{100 * npv:.2f}"
    V[f"T4_{key}_MISS"] = c((1 - se) * prev * 1e4); V[f"T4_{key}_FA"] = c((1 - sp) * (1 - prev) * 1e4)
V["T4_D_PREV"] = f"{100 * prev_data:.1f}"
ppv1 = se * .01 / (se * .01 + (1 - sp) * .99)
V["T4_1_FA_OF_10"] = str(int(round(10 * (1 - ppv1))))

V["DELTA_ACC"] = f"{97.53 - per_run['acc@0.5'].mean():.2f}"

with open(os.path.join(RESULTS_DIR, "paper_values.json"), "w") as f:
    json.dump(V, f, indent=1, ensure_ascii=False)
for k_, v_ in V.items():
    print(f"{k_:16s} {v_}")
print("\nSaved to", RESULTS_DIR)
print("Download paper_values.json, fig3_confusion.png and fig5_gradcam.png, then run fill_paper.py.")

## 11. Put the results into the paper
Upload `Heart_Disease_XAI_IEEE_Conference_revised.docx` and `fill_paper.py` to the results folder on Drive
(`MyDrive/heart_xai_results`), then run this cell. It writes `..._filled.docx` next to them.
Afterwards, open the filled paper in Word, write the three green notes (they need your own reading of
Fig. 5), remove their green highlight, and re-read the sentences around the new numbers.

In [ ]:
import subprocess
paper = os.path.join(RESULTS_DIR, "Heart_Disease_XAI_IEEE_Conference_revised.docx")
script = os.path.join(RESULTS_DIR, "fill_paper.py")
print(subprocess.run(["python", script, paper, RESULTS_DIR,
                      paper.replace("_revised.docx", "_filled.docx")],
                     capture_output=True, text=True).stdout)